# Les membres statiques
# Static members

- Un membre statique appartient à la **classe elle-même**, pas à une instance : une seule valeur, partagée par tous les objets
- A static member belongs to the **class itself**, not to an instance: a single value, shared by all the objects

<br>

- On y accède par le **nom de la classe** : `Voiture.nb_creees`
- It is accessed through the **class name**: `Voiture.nb_creees`

<br>

- ⚠️ Particularité de Python (contrairement à C# et Java) : un membre statique est **aussi accessible depuis une instance**
- ⚠️ Python specificity (unlike C# and Java): a static member is **also reachable through an instance**

## L'attribut de classe (statique)
## The class (static) attribute

- Déclaré **directement dans la classe**, en dehors du `__init__`
- Declared **directly in the class**, outside `__init__`

<br>

- Usage typique : un compteur d'instances, une constante partagée
- Typical use: an instance counter, a shared constant

In [ ]:
class Voiture:
    nb_creees = 0                      # attribut de CLASSE : une seule valeur partagée
    # CLASS attribute: a single shared value

    def __init__(self, marque):
        self.marque = marque           # attribut d'INSTANCE : propre à chaque objet
        # INSTANCE attribute: each object has its own
        Voiture.nb_creees += 1         # on incrémente via le NOM DE LA CLASSE
        # we increment through the CLASS NAME


voiture_1 = Voiture("Toyota")
voiture_2 = Voiture("Kia")
voiture_3 = Voiture("Tesla")

print(Voiture.nb_creees)       # 3 - lecture via la classe / read through the class
print(voiture_1.nb_creees)     # 3 - accessible aussi via l'instance (particularité Python)
# also reachable through the instance (Python specificity)

## Le piège du masquage
## The shadowing trap

- Affecter via l'instance ne modifie **pas** l'attribut de classe : cela crée un attribut d'instance qui le **masque** pour cet objet-là seulement
- Assigning through the instance does **not** change the class attribute: it creates an instance attribute that **shadows** it for that object only

<br>

- Règle de **lecture** : Python cherche d'abord dans l'instance, puis dans la classe
- **Read** rule: Python looks in the instance first, then in the class

<br>

- Règle d'**écriture** : `instance.x = ...` écrit **toujours** dans l'instance
- **Write** rule: `instance.x = ...` **always** writes into the instance

In [ ]:
class Compteur:
    total = 0                          # attribut de classe / class attribute


compteur_1 = Compteur()
compteur_1.total = 10                  # crée un attribut d'INSTANCE qui MASQUE total
# creates an INSTANCE attribute that SHADOWS total

print(Compteur.total)                  # 0  - l'attribut de classe n'a pas bougé
# the class attribute did not change
print(compteur_1.total)                # 10 - cette instance voit le sien
# this instance sees its own

Compteur.total = 99                    # modifier la valeur partagée : via la CLASSE
# to change the shared value: through the CLASS
print(Compteur.total)                  # 99

## `@staticmethod` : une méthode sans `self`
## `@staticmethod`: a method without `self`

- Une méthode statique n'utilise **aucune donnée d'instance** : pas de `self`
- A static method uses **no instance data**: no `self`

<br>

- Elle vit dans la classe parce qu'elle s'y rattache **logiquement** (fonction utilitaire liée au type)
- It lives in the class because it **logically** belongs to it (utility function related to the type)

In [ ]:
class Conversion:

    @staticmethod
    def km_en_miles(km):               # pas de self ! / no self!
        return round(km * 0.621371, 1)

    @staticmethod
    def miles_en_km(miles):
        return round(miles / 0.621371, 1)


print(Conversion.km_en_miles(100))     # 62.1 - appel via la classe / call through the class
print(Conversion.miles_en_km(62.1))    # 99.9

## `@classmethod` et `cls`
## `@classmethod` and `cls`

- Une méthode de classe reçoit **la classe elle-même** comme premier paramètre : `cls` (comme `self` reçoit l'instance)
- A class method receives **the class itself** as first parameter: `cls` (like `self` receives the instance)

<br>

- Son grand usage : les **constructeurs alternatifs** (créer un objet depuis un dictionnaire, un fichier, une chaîne...)
- Its main use: **alternative constructors** (build an object from a dictionary, a file, a string...)

In [ ]:
class Voiture:
    def __init__(self, marque, annee):
        self.marque = marque
        self.annee = annee

    @classmethod
    def depuis_dict(cls, donnees):     # cls = la classe (Voiture) / cls = the class (Voiture)
        return cls(donnees["marque"], donnees["annee"])


donnees = {"marque": "Kia", "annee": 2024}
voiture_1 = Voiture.depuis_dict(donnees)   # constructeur alternatif / alternative constructor
print(voiture_1.marque, voiture_1.annee)   # Kia 2024

## Synthèse
## Summary

| Décorateur / Decorator | 1er paramètre / 1st parameter | Accède à / Has access to |
|---|---|---|
| (aucun / none) | `self` : l'instance / the instance | les attributs de l'objet / the object's attributes |
| `@classmethod` | `cls` : la classe / the class | les attributs de classe, les constructeurs alternatifs / class attributes, alternative constructors |
| `@staticmethod` | aucun / none | rien : une fonction rangée dans la classe / nothing: a function stored in the class |

<br>

**Le bon réflexe pour choisir / How to choose:**

- La méthode utilise `self` → méthode d'instance (le cas normal)
- The method uses `self` → instance method (the normal case)

<br>

- Elle crée ou décrit **la classe** → `@classmethod`
- It creates or describes **the class** → `@classmethod`

<br>

- Elle n'utilise ni l'un ni l'autre → `@staticmethod`
- It uses neither → `@staticmethod`

<br>

> **Question d'entretien** : « Pourquoi `Voiture.nb_creees += 1` et pas `self.nb_creees += 1` dans le `__init__` ? » — Parce que `self.nb_creees += 1` créerait un attribut d'instance qui **masque** le compteur partagé : chaque voiture compterait... jusqu'à 1.
>
> **Interview question**: "Why `Voiture.nb_creees += 1` and not `self.nb_creees += 1` in `__init__`?" — Because `self.nb_creees += 1` would create an instance attribute that **shadows** the shared counter: each car would count... up to 1.